# Experiment 1: is the value-head damage avoidable?

The 3000-step sweep produced a clean trade. Consistency **helped** the conditioned policy and **hurt** the
value head, in exactly the same three settings:

| setting | act_kl | value_kl |
| --- | --- | --- |
| bin 10 | cons better (~0.055 vs 0.062) | mc better (~0.062 vs 0.09) |
| bin 11 | cons better (~0.050 vs 0.075) | mc better (~0.085 vs 0.12) |
| best far | cons better (~0.63 vs 0.68) | mc better (~0.080 vs 0.10) |

The mechanism is available: `delta_t = v_t - u_t + b_{t-1} - b_t` has **no stop-gradient anywhere**, by
design (note section 6 -- neither side of an interval identity is presumed a correct teacher). So the
identity can be satisfied by dragging the value head toward the token heads instead of the reverse. And
`value_kl` on the conditioned settings reads the NOR value head on prefixes from the *conditioned* process,
which the random-walk MC supervision barely covers -- so off-distribution, the consistency pull wins.

**This notebook tests whether that trade is necessary.** `cons_detach` stop-gradients one whole term:

- `none` -- the current behaviour, gradient into everything
- `b` -- the value head is a **fixed teacher**; only the token heads move to satisfy the identity
- `uv` -- the mirror image: token heads fixed, only the value head moves

Detaching a whole *term* is well defined under the variance shortcut, because it happens before `delta` and
`c` are formed. Per-interval teacher/target detachment is **not** -- each `c_k` is the later endpoint of some
pairs and the earlier endpoint of others, so "detach the target side of each pair" has no expression in terms
of the `c_k` (note section 6). That is why the flag detaches terms, not sides.

**What would make this a win:** `b` keeps the act_kl gain on bins 10/11/best far *and* recovers baseline
value_kl. **What would kill the idea:** `b` loses the act_kl gain too, meaning the gain was coming from
moving the value head, not from better conditioning.

Using `local` and `all_scaled` as the two objectives -- the last sweep showed the choice barely matters
(`rms Delta ~ l^0.14`), so two is enough to check the effect is not objective-specific.

In [ ]:
REPO = "https://github.com/amdson/scrl.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
RUNS_DIR = "/content/drive/MyDrive/sillyrl/runs"  #@param {type:"string"}

In [ ]:
# Clone (or update) the repo and make sure the dependencies are importable.
import os, subprocess, sys

url = REPO
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret: public repo

if not os.path.exists("/content/sillyrl/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/sillyrl"], check=True)
else:
    subprocess.run(["git", "-C", "/content/sillyrl", "pull", "-q"], check=True)
os.chdir("/content/sillyrl")
sys.path.insert(0, "/content/sillyrl")

# Colab's preinstalled flax can lag its JAX (e.g. flax calling jax.core APIs that JAX 0.11 removed), so always
# upgrade flax and optax before importing them. pip keeps Colab's JAX if it already satisfies them; if it had
# to upgrade JAX, bring the GPU plugin (jax-cuda*) to the same version so the runtime doesn't fall back to CPU.
import importlib.metadata as md
jax_before = md.version("jax")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "flax", "optax"], check=True)
jax_after = md.version("jax")
if jax_after != jax_before:
    plugins = sorted({d.metadata["Name"] for d in md.distributions()
                      if (d.metadata["Name"] or "").lower().replace("_", "-").startswith("jax-cuda")})
    if plugins:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={jax_after}" for p in plugins]], check=True)
    print(f"jax {jax_before} -> {jax_after}; plugins updated: {plugins}")

import jax, flax, optax
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__, "|", jax.devices())

In [ ]:
# Where runs are saved. Must be set before importing maze_consistency.train.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RUNS_DIR"] = RUNS_DIR
else:
    os.environ["RUNS_DIR"] = "runs"
os.makedirs(os.environ["RUNS_DIR"], exist_ok=True)
print("runs ->", os.environ["RUNS_DIR"])

In [ ]:
# Rebuild the dataset (100k random walks) and the exact test set from data/canonical/maze.txt. Deterministic.
!python run.py dataset | tail -4
!python run.py testset | tail -1

## 1. The loss configurations

Seven runs: the `mc` baseline plus two objectives x three detach settings.

In [ ]:
from dataclasses import replace, asdict
from maze_consistency.train import LossConfig
import maze_consistency.consistency as C

CONS_BATCH   = 16
LAMBDA       = {"local": 0.15, "all_scaled": 0.14}
DETACH       = ("none", "b", "uv")

LOSSES = {"mc": LossConfig(mc=True)}
for obj, lam in LAMBDA.items():
    for det in DETACH:
        LOSSES[f"{obj}_{det}"] = LossConfig(mc=True, cons=True, cons_loss=obj, w_cons=lam,
                                            cons_batch=CONS_BATCH, cons_detach=det)

for k, lc in LOSSES.items():
    print(f"  {k:<18} cons={lc.cons!s:<5} loss={lc.cons_loss if lc.cons else '-':<12} "
          f"lambda={lc.w_cons if lc.cons else 0:<7g} detach={lc.cons_detach}")

## 2. The sweep

Same loop as `consistency_sweep.ipynb`: every parameter is an argument, edit the body in place.

In [ ]:
import os
import numpy as np
from maze_consistency.dataset import load as load_data
from maze_consistency.dp import compute_ground_truth
from maze_consistency.tokens import Tokenizer
from maze_consistency.model import ModelConfig, MazeTransformer
from maze_consistency.testset import load_testset, stratified_rows, score
from maze_consistency.train import train, load_run, RUNS_DIR, N_HELDOUT

MAZE, DATA = load_data()
TOK = Tokenizer(MAZE)
N_TRAIN = len(DATA["length"]) - N_HELDOUT

# One fixed set of held-out rollouts, shared by every run, so the consistency numbers compare directly.
HELDOUT = np.random.default_rng(0).choice(np.arange(N_TRAIN, N_TRAIN + N_HELDOUT), 64, replace=False)


def run_sweep(losses, seeds=(0,), steps=3000, batch=32, lr=1e-3, d_model=64, n_layers=2, n_heads=4,
              eval_every=250, eval_per_setting=50, heldout=HELDOUT, log_every=100, prefix="cons",
              skip_existing=True, log=print):
    """Train every (loss config, seed) pair into RUNS_DIR/<prefix>/<name>_s<seed>/.

    losses            {run name: LossConfig}
    seeds             one run per seed; the plots below show a min..max band when there are several
    eval_per_setting  exact-test rows scored per setting at each checkpoint (same rows for every run)
    heldout           rollouts the consistency losses are measured on at each checkpoint (same for every run)
    skip_existing     leave finished runs alone, so a disconnected session resumes where it stopped
    """
    ts = load_testset()
    rows = stratified_rows(ts, eval_per_setting, seed=0)
    cfg = ModelConfig.for_tokenizer(TOK, d_model=d_model, n_layers=n_layers, n_heads=n_heads)
    cons_eval = C.make_heldout_eval(MazeTransformer(cfg), TOK, MAZE, DATA, heldout)

    def eval_fn(params, fwd):                      # edit to track whatever you want; keys become the history
        m = score(params, fwd, TOK, ts, rows)      # act_kl / value_kl / start_kl / dyn_nll vs the exact DP
        m.pop("per_setting")
        m.update(cons_eval(params))                # cons/<objective> and diag/*, identical batch every run
        return m

    done = {}
    for name, lc in losses.items():
        for seed in seeds:
            run = f"{prefix}/{name}_s{seed}"
            if skip_existing and os.path.exists(os.path.join(RUNS_DIR, run, "history.json")):
                log(f"[skip] {run} exists")
                continue
            done[run] = train(name=run, steps=steps, batch=batch, lr=lr, d_model=d_model, n_layers=n_layers,
                              n_heads=n_heads, seed=seed, loss=lc, eval_fn=eval_fn, eval_every=eval_every,
                              log_every=log_every, log=log)
    return done


PREFIX = "cons"
run_sweep(LOSSES, seeds=(0,), steps=3000, batch=32, eval_every=250, eval_per_setting=50, prefix=PREFIX)

## 3. Exact-test metrics

`act_kl` is KL(true action distribution || model), `value_kl` the same for the value head read in NOR mode.
Note `act_kl/NOR` is a near-trivial target -- the true NOR policy is uniform 1/4 -- so read the conditioned
settings (`bin 10`, `bin 11`, `best far`) as the real ones.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt


def load_history(prefix, which="test"):
    """{run name: [history per seed]}. which="test" -> exact-test metrics at each checkpoint;
    which="train" -> logged loss parts, including cons / cond_gap / info_gain."""
    root = os.path.join(RUNS_DIR, prefix)
    out = {}
    for d in sorted(os.listdir(root)) if os.path.isdir(root) else []:
        p = os.path.join(root, d, "history.json")
        if os.path.exists(p):
            with open(p) as f:
                out.setdefault(d.rsplit("_s", 1)[0], []).append(json.load(f)[which])
    return out


def plot_curves(prefix, metrics=("act_kl", "value_kl"),
                settings=("NOR", "bin 10", "bin 11", "best far"), logy=True, order=None, figsize=(4.2, 3.4)):
    runs = load_history(prefix, "test")
    names = [n for n in (order or LOSSES) if n in runs] or list(runs)
    fig, ax = plt.subplots(len(metrics), len(settings),
                           figsize=(figsize[0] * len(settings), figsize[1] * len(metrics)), squeeze=False)
    for i, metric in enumerate(metrics):
        for j, setting in enumerate(settings):
            a, key = ax[i, j], f"{metric}/{setting}"
            for c, name in enumerate(names):
                hists = runs[name]
                steps = [m["step"] for m in hists[0]]
                ys = np.array([[m.get(key, np.nan) for m in h] for h in hists], dtype=float)
                a.plot(steps, np.nanmean(ys, 0), color=f"C{c}", label=name)
                if len(hists) > 1:
                    a.fill_between(steps, np.nanmin(ys, 0), np.nanmax(ys, 0), color=f"C{c}", alpha=.15)
            if logy:
                a.set_yscale("log")
            a.set_title(key, fontsize=9); a.set_xlabel("step"); a.grid(alpha=.3)
    ax[0, 0].legend(fontsize=8)
    fig.tight_layout()
    return fig


plot_curves(PREFIX)
plt.show()

## 4. Collapse diagnostics

`cond_gap` -> 0 means the conditioned and unconditioned policies have merged; `info_gain` -> 0 means the
reward head has gone flat. Either decaying while `cons` falls = lambda too high.

In [ ]:
def plot_diagnostics(prefix, keys=(("tf", "data: teacher-forced next-token loss", True),
                                   ("cons", "consistency objective", True),
                                   ("cond_gap", "cond_gap: mean |v_t - u_t|   (-> 0 = R ignored)", False),
                                   ("info_gain", "info_gain: log q_n(R) - log q_0(R)   (-> 0 = head flat)", False)),
                     order=None, figsize=(4.6, 3.8)):
    """Train-side view. keys is (history key, panel title, log y). Configs with no consistency term simply
    do not appear in the cons/cond_gap/info_gain panels."""
    runs = load_history(prefix, "train")
    names = [n for n in (order or LOSSES) if n in runs] or list(runs)
    fig, ax = plt.subplots(1, len(keys), figsize=(figsize[0] * len(keys), figsize[1]), squeeze=False)
    for j, (key, title, logy) in enumerate(keys):
        a = ax[0, j]
        for c, name in enumerate(names):
            hists = runs[name]
            steps = [m["step"] for m in hists[0]]
            ys = np.array([[m.get(key, np.nan) for m in h] for h in hists], dtype=float)
            if np.isnan(ys).all():
                continue
            a.plot(steps, np.nanmean(ys, 0), color=f"C{c}", label=name)
            if len(hists) > 1:
                a.fill_between(steps, np.nanmin(ys, 0), np.nanmax(ys, 0), color=f"C{c}", alpha=.15)
        a.set_yscale("log") if logy else a.axhline(0, color="k", lw=.8, ls=":")
        a.set_title(title, fontsize=9); a.set_xlabel("step"); a.grid(alpha=.3)
    ax[0, 0].legend(fontsize=7)
    fig.tight_layout()
    return fig


plot_diagnostics(PREFIX)
plt.show()

## 5. Held-out consistency against the exact model

All six objectives on one fixed held-out batch, identical for every run. The true model scores 0.

In [ ]:
GT = compute_ground_truth(MAZE)
exact, _ = C.exact_losses(MAZE, GT, DATA, HELDOUT)
print("the true model on the same held-out rollouts (0 = perfectly consistent):")
print("  " + "  ".join(f"{k} {float(np.asarray(v).mean()):.2e}" for k, v in exact.items()))

plot_curves(PREFIX, metrics=("cons",), settings=tuple(C.ALL), figsize=(3.4, 3.2))
plt.show()
plot_curves(PREFIX, metrics=("diag",), settings=("cond_gap", "info_gain", "drift"), logy=False)
plt.show()

## 6. Final numbers

In [ ]:
def final_table(prefix, baseline="mc", metrics=("act_kl", "value_kl"),
                settings=("NOR", "bin 10", "bin 11", "best far"),
                diag=("cons/local", "cons/all_scaled", "cons/multiscale", "diag/cond_gap", "diag/info_gain"),
                order=None):
    test = load_history(prefix, "test")
    cols = [f"{m}/{s}" for m in metrics for s in settings]
    names = [n for n in (order or LOSSES) if n in test] or list(test)
    mean_last = lambda hs, k: float(np.mean([h[-1].get(k, np.nan) for h in hs]))
    rows = {n: {k: mean_last(test[n], k) for k in cols + list(diag)} for n in names}
    base = rows.get(baseline)
    w = max(len(n) for n in rows) + 2

    print(f"exact-test KL in nats, lower is better. (d) = minus `{baseline}`, so negative beats baseline.")
    for m in metrics:
        group = [f"{m}/{s}" for s in settings]
        print()
        print(" " * w + "".join(f"{k.split('/')[1]:>22}" for k in group))
        print(f"{m:<{w}}" + "".join(f"{'value':>12}{'(d)':>10}" for _ in group))
        for n, r in rows.items():
            line = "".join(f"{r[k]:12.4f}" + ("".rjust(10) if base is None or n == baseline
                                              else f"{r[k] - base[k]:+10.4f}") for k in group)
            print(f"{n:<{w}}" + line)
    print()
    print("held out, same rollouts for every run; the true model scores 0 on every cons/ column")
    print(f"{'config':<{w}}" + "".join(f"{k:>18}" for k in diag))
    for n, r in rows.items():
        print(f"{n:<{w}}" + "".join(f"{r[k]:18.4f}" if np.isfinite(r[k]) else f"{'-':>18}" for k in diag))
    return rows


_ = final_table(PREFIX)

## 7. The verdict

The question in one table. For each objective, the baseline against the three detach settings, on the
settings where the trade showed up. `act_kl` wants to stay **below** `mc`; `value_kl` wants to come back
**down to** `mc`.

In [ ]:
def detach_verdict(prefix=PREFIX, baseline="mc", objectives=("local", "all_scaled"),
                   detach=("none", "b", "uv"), settings=("bin 10", "bin 11", "best far")):
    test = load_history(prefix, "test")
    last = lambda n, k: float(np.mean([h[-1].get(k, np.nan) for h in test[n]])) if n in test else np.nan
    for metric, want in (("act_kl", "lower than mc = the gain survived"),
                         ("value_kl", "lower than mc = the damage is undone")):
        print(f"{metric}   ({want})")
        print(f"  {'run':<18}" + "".join(f"{s:>14}" for s in settings))
        print(f"  {baseline:<18}" + "".join(f"{last(baseline, f'{metric}/{s}'):14.4f}" for s in settings))
        for obj in objectives:
            for det in detach:
                n = f"{obj}_{det}"
                cells = ""
                for s in settings:
                    v, b = last(n, f"{metric}/{s}"), last(baseline, f"{metric}/{s}")
                    cells += (f"{v:13.4f}{'*' if v < b else ' '}" if np.isfinite(v) else f"{'-':>14}")
                print(f"  {n:<18}" + cells)
        print()
    print("* = better than the mc baseline")


detach_verdict()